In [ ]:
# set paths and links

import duckdb
from pathlib import Path

Path("../../data/generated/CAR/duckdb_temp").mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

con.execute("SET memory_limit='3GB'")
con.execute("SET temp_directory='../../data/generated/CAR/duckdb_temp'")

ROOT = Path("../..")

sample_path = ROOT / "data/generated/tone/full_10k_sample_dedup_stratified_1500_per_year.csv"
ret_path = ROOT / "data/external/ret_all.csv.gz"
index_path = ROOT / "data/external/index.csv"
link_path = ROOT / "data/external/cik_to_permno.csv.gz"

out_path = ROOT / "data/generated/CAR/10k_sample_with_car.csv"

print("sample:", sample_path.exists(), sample_path)
print("returns:", ret_path.exists(), ret_path)
print("index:", index_path.exists(), index_path)
print("link:", link_path.exists(), link_path)

sample: True ../../data/generated/full_10k_sample_dedup_stratified_1500_per_year.csv
returns: True ../../data/external/ret_all.csv.gz
index: True ../../data/external/index.csv
link: True ../../data/external/cik_to_permno.csv.gz


In [4]:
# check the sample data

con.execute(f"""
SELECT *
FROM read_csv_auto('{sample_path}', all_varchar=true)
LIMIT 5
""").df()

,cik,name,tickers,exchanges,entity_type,sic,state_of_incorporation,filing_date,report_date,url,file_size_in_bytes,download_success,download_error,word_count,report_year,original_row_number,reason_type,dedup_action,dedup_rule,accession_number
0,0000949956,VERITY INC \DE\,NaN,NaN,operating,7374.0,DE,2002-08-16,2002-05-31,https://www.sec.gov/Archives/edgar/data/949956...,1015648,True,None,33496.0,2002,50855,None,kept,kept_unflagged_or_reason1,0000891618-02-003972
1,0001464830,"Silver Falcon Mining, Inc.",NaN,NaN,operating,1040.0,DE,2012-04-11,2011-12-31,https://www.sec.gov/Archives/edgar/data/146483...,2496075,True,None,35798.0,2011,124429,None,kept,kept_unflagged_or_reason1,0001091818-12-000111
2,0001141240,LIQUIDMETAL TECHNOLOGIES INC,LQMT,OTC,operating,2800.0,DE,2003-03-31,2002-12-31,https://www.sec.gov/Archives/edgar/data/114124...,1021000,True,None,38069.0,2002,77735,None,kept,kept_unflagged_or_reason1,0000950144-03-004158
3,0001053112,CABLEVISION SYSTEMS CORP /NY,NaN,NaN,operating,4841.0,DE,2014-02-26,2013-12-31,https://www.sec.gov/Archives/edgar/data/105311...,29772959,True,None,86475.0,2013,61707,None,kept,kept_unflagged_or_reason1,0001140361-14-009776
4,0001286243,INTERSTAR MILLENNIUM SERIES 2002-1G TRUST,NaN,NaN,operating,6189.0,NaN,2004-10-19,2003-12-31,https://www.sec.gov/Archives/edgar/data/128624...,65841,True,None,2739.0,2003,91603,None,kept,kept_unflagged_or_reason1,0000902561-04-000463


In [5]:
# check the returns data

con.execute(f"""
SELECT *
FROM read_csv_auto('{ret_path}', all_varchar=true)
LIMIT 5
""").df()

,PERMNO,date,SHRCD,EXCHCD,TICKER,PERMCO,CUSIP,PRC,VOL,RET,SHROUT,RETX
0,10001,2000-01-03,11,3,EWST,7953,36720410,8.56250,1721,0.007353,2450,0.007353
1,10001,2000-01-04,11,3,EWST,7953,36720410,8.43750,1080,-0.014599,2450,-0.014599
2,10001,2000-01-05,11,3,EWST,7953,36720410,8.56250,1711,0.014815,2450,0.014815
3,10001,2000-01-06,11,3,EWST,7953,36720410,8.50000,580,-0.007299,2450,-0.007299
4,10001,2000-01-07,11,3,EWST,7953,36720410,8.43750,1406,-0.007353,2450,-0.007353


In [6]:
# check the market index data

con.execute(f"""
SELECT *
FROM read_csv_auto('{index_path}', all_varchar=true)
LIMIT 5
""").df()

,DATE,vwretd,ewretd
0,2000-01-03,-0.006803,0.002878
1,2000-01-04,-0.039652,-0.017465
2,2000-01-05,-0.000935,0.007821
3,2000-01-06,-0.007391,0.004504
4,2000-01-07,0.032516,0.017008


In [27]:
# set up the sample data with proper types and row numbers

con.execute(f"""
CREATE OR REPLACE TEMP TABLE sample AS
SELECT
    ROW_NUMBER() OVER () AS row_id,
    *,
    TRY_CAST(cik AS BIGINT) AS cik_num,
    TRY_CAST(filing_date AS DATE) AS filing_dt,
    TRY_CAST(report_date AS DATE) AS report_dt
FROM read_csv_auto(
    '{sample_path}',
    all_varchar = true
)
WHERE filing_date IS NOT NULL
  AND TRY_CAST(filing_date AS DATE) IS NOT NULL
  AND TRY_CAST(filing_date AS DATE) <= DATE '2024-12-31'
""")

In [28]:
# check the results of the sample data

con.execute("""
SELECT
    row_id,
    cik,
    cik_num,
    name,
    filing_date,
    filing_dt,
    report_date,
    report_dt
FROM sample
LIMIT 10
""").df()

,row_id,cik,cik_num,name,filing_date,filing_dt,report_date,report_dt
0,1,0000949956,949956,VERITY INC \DE\,2002-08-16,2002-08-16,2002-05-31,2002-05-31
1,2,0001464830,1464830,"Silver Falcon Mining, Inc.",2012-04-11,2012-04-11,2011-12-31,2011-12-31
2,3,0001141240,1141240,LIQUIDMETAL TECHNOLOGIES INC,2003-03-31,2003-03-31,2002-12-31,2002-12-31
3,4,0001053112,1053112,CABLEVISION SYSTEMS CORP /NY,2014-02-26,2014-02-26,2013-12-31,2013-12-31
4,5,0001286243,1286243,INTERSTAR MILLENNIUM SERIES 2002-1G TRUST,2004-10-19,2004-10-19,2003-12-31,2003-12-31
5,6,0001304973,1304973,Alternative Loan Trust 2004-J9,2005-03-29,2005-03-29,2004-12-31,2004-12-31
6,7,0001051343,1051343,COMMUNITY WEST BANCSHARES /,2009-03-30,2009-03-30,2008-12-31,2008-12-31
7,8,0000753557,753557,LBO CAPITAL CORP,2009-04-15,2009-04-15,2008-12-31,2008-12-31
8,9,0001168990,1168990,"SUPERFUND GREEN, L.P.",2003-03-31,2003-03-31,2002-12-31,2002-12-31
9,10,0001378125,1378125,Toga Ltd,2010-10-29,2010-10-29,2010-07-31,2010-07-31


In [31]:
# check data range of the sample filings

con.execute("""
SELECT
    COUNT(*) AS n_filings,
    MIN(filing_dt) AS min_filing_date,
    MAX(filing_dt) AS max_filing_date
FROM sample
""").df()

,n_filings,min_filing_date,max_filing_date
0,33236,2002-03-26,2024-12-31


In [32]:
# set up returns data

con.execute(f"""
CREATE OR REPLACE TEMP TABLE ret AS
SELECT
    TRY_CAST(PERMNO AS BIGINT) AS permno,
    TRY_CAST(date AS DATE) AS ret_date,
    TRY_CAST(RET AS DOUBLE) AS ret,
    TRY_CAST(RETX AS DOUBLE) AS retx,
    TRY_CAST(PRC AS DOUBLE) AS prc,
    TRY_CAST(VOL AS DOUBLE) AS vol,
    TRY_CAST(SHRCD AS INTEGER) AS shrcd,
    TRY_CAST(EXCHCD AS INTEGER) AS exchcd
FROM read_csv_auto(
    '{ret_path}',
    all_varchar = true
)
WHERE TRY_CAST(PERMNO AS BIGINT) IS NOT NULL
  AND TRY_CAST(date AS DATE) IS NOT NULL
  AND TRY_CAST(RET AS DOUBLE) IS NOT NULL
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [33]:
# check the results of the returns data

con.execute("""
SELECT *
FROM ret
LIMIT 10
""").df()

,permno,ret_date,ret,retx,prc,vol,shrcd,exchcd
0,10001,2000-01-03,0.007353,0.007353,8.56250,1721.0,11,3
1,10001,2000-01-04,-0.014599,-0.014599,8.43750,1080.0,11,3
2,10001,2000-01-05,0.014815,0.014815,8.56250,1711.0,11,3
3,10001,2000-01-06,-0.007299,-0.007299,8.50000,580.0,11,3
4,10001,2000-01-07,-0.007353,-0.007353,8.43750,1406.0,11,3
5,10001,2000-01-10,0.000000,0.000000,8.43750,3390.0,11,3
6,10001,2000-01-11,0.003704,0.003704,-8.46875,0.0,11,3
7,10001,2000-01-12,-0.011070,-0.011070,8.37500,14025.0,11,3
8,10001,2000-01-13,-0.029851,-0.029851,8.12500,1700.0,11,3
9,10001,2000-01-14,0.030769,0.030769,8.37500,850.0,11,3


In [34]:
# check the date range of the returns data

con.execute("""
SELECT
    MIN(ret_date) AS min_date,
    MAX(ret_date) AS max_date,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT permno) AS n_permno
FROM ret
""").df()

,min_date,max_date,n_rows,n_permno
0,2000-01-03,2024-12-31,25402296,8432


In [35]:
# set up market index data

con.execute(f"""
CREATE OR REPLACE TEMP TABLE mkt AS
SELECT
    TRY_CAST(DATE AS DATE) AS ret_date,
    TRY_CAST(vwretd AS DOUBLE) AS vwretd,
    TRY_CAST(ewretd AS DOUBLE) AS ewretd
FROM read_csv_auto(
    '{index_path}',
    all_varchar = true
)
WHERE TRY_CAST(DATE AS DATE) IS NOT NULL
  AND TRY_CAST(vwretd AS DOUBLE) IS NOT NULL
""")

In [36]:
# check the results of the market index data

con.execute("""
SELECT *
FROM mkt
LIMIT 10
""").df()

,ret_date,vwretd,ewretd
0,2000-01-03,-0.006803,0.002878
1,2000-01-04,-0.039652,-0.017465
2,2000-01-05,-0.000935,0.007821
3,2000-01-06,-0.007391,0.004504
4,2000-01-07,0.032516,0.017008
5,2000-01-10,0.018608,0.015333
6,2000-01-11,-0.016941,-0.006079
7,2000-01-12,-0.006922,-0.002505
8,2000-01-13,0.016265,0.013063
9,2000-01-14,0.011137,0.009119


In [37]:
# check the date range of the market index data

con.execute("""
SELECT
    MIN(ret_date) AS min_date,
    MAX(ret_date) AS max_date,
    COUNT(*) AS n_days
FROM mkt
""").df()

,min_date,max_date,n_days
0,2000-01-03,2024-12-31,6289


In [38]:
# set up cik to permno link data

con.execute(f"""
CREATE OR REPLACE TEMP TABLE link AS
SELECT DISTINCT
    TRY_CAST(cik AS BIGINT) AS cik_num,
    TRY_CAST(LPERMNO AS BIGINT) AS permno,
    TRY_CAST(LPERMCO AS BIGINT) AS permco,
    GVKEY AS gvkey,
    LINKTYPE AS linktype,
    TRY_CAST(LINKDT AS DATE) AS linkdt,
    CASE
        WHEN LINKENDDT IS NULL OR LINKENDDT = '' OR LINKENDDT = 'E'
            THEN DATE '9999-12-31'
        ELSE TRY_CAST(LINKENDDT AS DATE)
    END AS linkenddt
FROM read_csv_auto(
    '{link_path}',
    all_varchar = true,
    ignore_errors = true
)
WHERE TRY_CAST(cik AS BIGINT) IS NOT NULL
  AND TRY_CAST(LPERMNO AS BIGINT) IS NOT NULL
  AND TRY_CAST(LINKDT AS DATE) IS NOT NULL
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [39]:
# check the results of the link data

con.execute("""
SELECT *
FROM link
LIMIT 10
""").df()

,cik_num,permno,permco,gvkey,linktype,linkdt,linkenddt
0,61478,50906,2902,001013,LU,1979-03-16,2010-12-31
1,859163,81912,30930,001072,LC,1995-08-15,2020-03-27
2,764622,27991,21409,001075,LU,1962-01-31,9999-12-31
3,1808834,10517,5674,001076,LC,2010-12-01,9999-12-31
4,1808834,10517,5674,001076,LC,1993-01-01,2010-11-30
5,1800,20482,20017,001078,LC,1962-01-31,9999-12-31
6,1923,10568,5580,001082,LC,1982-06-29,2011-08-31
7,2034,10656,37,001094,LC,1972-12-14,2019-10-02
8,2186,10779,65,001117,LC,2005-10-14,9999-12-31
9,2969,28222,20030,001209,LC,1962-01-31,9999-12-31


In [40]:
# check the size of the link data

con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT cik_num) AS n_cik,
    COUNT(DISTINCT permno) AS n_permno
FROM link
""").df()

,n_rows,n_cik,n_permno
0,8812,8319,8572


In [41]:
# match every filing to a permno using the link table

con.execute("""
CREATE OR REPLACE TEMP TABLE sample_linked AS
SELECT
    s.*,
    l.permno,
    l.permco,
    l.gvkey,
    l.linktype,
    l.linkdt,
    l.linkenddt
FROM sample AS s
LEFT JOIN link AS l
    ON s.cik_num = l.cik_num
   AND s.filing_dt >= l.linkdt
   AND s.filing_dt <= l.linkenddt
""")

In [42]:
# check the matching results

con.execute("""
SELECT
    COUNT(*) AS n_rows_after_link,
    COUNT(DISTINCT row_id) AS n_filings,
    COUNT(DISTINCT CASE WHEN permno IS NOT NULL THEN row_id END) AS n_filings_with_permno,
    COUNT(DISTINCT CASE WHEN permno IS NOT NULL THEN row_id END) * 1.0
        / COUNT(DISTINCT row_id) AS filing_match_rate
FROM sample_linked
""").df()

,n_rows_after_link,n_filings,n_filings_with_permno,filing_match_rate
0,33351,33236,13757,0.413919


In [43]:
# check the filings that did not match to a permno

con.execute("""
SELECT
    row_id,
    cik,
    name,
    filing_date,
    report_date
FROM sample_linked
WHERE permno IS NULL
LIMIT 20
""").df()

,row_id,cik,name,filing_date,report_date
0,1,0000949956,VERITY INC \DE\,2002-08-16,2002-05-31
1,2,0001464830,"Silver Falcon Mining, Inc.",2012-04-11,2011-12-31
2,3,0001141240,LIQUIDMETAL TECHNOLOGIES INC,2003-03-31,2002-12-31
3,4,0001053112,CABLEVISION SYSTEMS CORP /NY,2014-02-26,2013-12-31
4,5,0001286243,INTERSTAR MILLENNIUM SERIES 2002-1G TRUST,2004-10-19,2003-12-31
5,6,0001304973,Alternative Loan Trust 2004-J9,2005-03-29,2004-12-31
6,8,0000753557,LBO CAPITAL CORP,2009-04-15,2008-12-31
7,9,0001168990,"SUPERFUND GREEN, L.P.",2003-03-31,2002-12-31
8,10,0001378125,Toga Ltd,2010-10-29,2010-07-31
9,12,0001393540,IGEN NETWORKS CORP,2022-03-31,2021-12-31


In [44]:
# check the filings that matched to multiple permnos

con.execute("""
SELECT
    row_id,
    cik,
    name,
    filing_date,
    COUNT(DISTINCT permno) AS n_permno
FROM sample_linked
WHERE permno IS NOT NULL
GROUP BY row_id, cik, name, filing_date
HAVING COUNT(DISTINCT permno) > 1
ORDER BY n_permno DESC
LIMIT 20
""").df()

,row_id,cik,name,filing_date,n_permno
0,27023,0001553079,"Empire State Realty OP, L.P.",2024-02-28,3
1,16500,0001553079,"Empire State Realty OP, L.P.",2016-02-26,3
2,1008,0001553079,"Empire State Realty OP, L.P.",2014-03-24,3
3,16498,0001553079,"Empire State Realty OP, L.P.",2022-02-25,3
4,25117,0001570585,Liberty Global Ltd.,2022-02-17,3
5,2454,0000729580,BEL FUSE INC /NJ,2010-03-12,2
6,10382,0000014693,BROWN FORMAN CORP,2022-06-17,2
7,3216,0000105016,WATSCO INC,2019-02-28,2
8,3739,0001029800,URSTADT BIDDLE PROPERTIES INC,2019-01-11,2
9,10626,0001669600,"Liberty Expedia Holdings, Inc.",2019-02-08,2


In [55]:
# extract target permnos that are needed for this sample

con.execute("""
CREATE OR REPLACE TEMP TABLE target_permnos AS
SELECT DISTINCT permno
FROM sample_linked
WHERE permno IS NOT NULL
""")

In [57]:
# check the target permnos

con.execute("""
SELECT
    COUNT(*) AS n_target_permnos
FROM target_permnos
""").df()

,n_target_permnos
0,6105


In [53]:
# get date bounds for the sample filings

con.execute("""
CREATE OR REPLACE TEMP TABLE date_bounds AS
SELECT
    MIN(filing_dt) - INTERVAL 10 DAY AS min_needed_date,
    MAX(filing_dt) + INTERVAL 10 DAY AS max_needed_date
FROM sample_linked
WHERE filing_dt IS NOT NULL
""")

In [54]:
# check the date bounds

con.execute("""
SELECT *
FROM date_bounds
""").df()

,min_needed_date,max_needed_date
0,2002-03-16,2025-01-10


In [58]:
# extract returns data for the target permnos and date bounds

con.execute(f"""
CREATE OR REPLACE TEMP TABLE ret_filtered AS
SELECT
    TRY_CAST(PERMNO AS BIGINT) AS permno,
    TRY_CAST(date AS DATE) AS ret_date,
    TRY_CAST(RET AS DOUBLE) AS ret,
    TRY_CAST(RETX AS DOUBLE) AS retx,
    TRY_CAST(PRC AS DOUBLE) AS prc,
    TRY_CAST(VOL AS DOUBLE) AS vol,
    TRY_CAST(SHRCD AS INTEGER) AS shrcd,
    TRY_CAST(EXCHCD AS INTEGER) AS exchcd
FROM read_csv_auto(
    '{ret_path}',
    all_varchar = true
) AS r
WHERE TRY_CAST(PERMNO AS BIGINT) IN (
    SELECT permno FROM target_permnos
)
AND TRY_CAST(date AS DATE) BETWEEN
    (SELECT min_needed_date FROM date_bounds)
    AND
    (SELECT max_needed_date FROM date_bounds)
AND TRY_CAST(RET AS DOUBLE) IS NOT NULL
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [59]:
# check filtered return data

con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT permno) AS n_permno,
    MIN(ret_date) AS min_ret_date,
    MAX(ret_date) AS max_ret_date
FROM ret_filtered
""").df()

,n_rows,n_permno,min_ret_date,max_ret_date
0,19786721,6105,2002-03-18,2024-12-31


In [60]:
# create market return table

con.execute(f"""
CREATE OR REPLACE TEMP TABLE mkt AS
SELECT
    TRY_CAST(DATE AS DATE) AS ret_date,
    TRY_CAST(vwretd AS DOUBLE) AS vwretd,
    TRY_CAST(ewretd AS DOUBLE) AS ewretd
FROM read_csv_auto(
    '{index_path}',
    all_varchar = true
)
WHERE TRY_CAST(DATE AS DATE) IS NOT NULL
  AND TRY_CAST(vwretd AS DOUBLE) IS NOT NULL
""")

In [61]:
# check market return data

con.execute("""
SELECT
    COUNT(*) AS n_days,
    MIN(ret_date) AS min_date,
    MAX(ret_date) AS max_date
FROM mkt
""").df()

,n_days,min_date,max_date
0,6289,2000-01-03,2024-12-31


In [62]:
# merge stock returns with market returns

con.execute("""
CREATE OR REPLACE TEMP TABLE ret_mkt AS
SELECT
    r.permno,
    r.ret_date,
    r.ret,
    r.retx,
    r.prc,
    r.vol,
    r.shrcd,
    r.exchcd,
    m.vwretd,
    m.ewretd,
    r.ret - m.vwretd AS abret_vw,
    r.ret - m.ewretd AS abret_ew
FROM ret_filtered AS r
LEFT JOIN mkt AS m
    ON r.ret_date = m.ret_date
WHERE r.ret IS NOT NULL
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [63]:
# check merged return data

con.execute("""
SELECT *
FROM ret_mkt
LIMIT 10
""").df()

,permno,ret_date,ret,retx,prc,vol,shrcd,exchcd,vwretd,ewretd,abret_vw,abret_ew
0,10001,2002-03-18,0.026569,0.026569,10.4710,3884.0,11,3,0.001273,0.004768,0.025296,0.021801
1,10001,2002-03-19,-0.001623,-0.001623,10.4540,2000.0,11,3,0.003372,0.002588,-0.004995,-0.004211
2,10001,2002-03-20,0.003405,0.003405,10.4896,3655.0,11,3,-0.014411,-0.005624,0.017816,0.009029
3,10001,2002-03-21,-0.026655,-0.026655,10.2100,2104.0,11,3,0.003144,0.006353,-0.029799,-0.033008
4,10001,2002-03-22,0.027424,0.027424,10.4900,4649.0,11,3,-0.004248,-0.000430,0.031672,0.027854
5,10001,2002-03-25,-0.013346,-0.013346,10.3500,700.0,11,3,-0.014346,-0.007033,0.001000,-0.006313
6,10001,2002-03-26,0.004783,0.004783,10.3995,2666.0,11,3,0.005784,0.004705,-0.001001,0.000078
7,10001,2002-03-27,0.009664,0.009664,10.5000,14710.0,11,3,0.005506,0.005686,0.004158,0.003978
8,10001,2002-03-28,-0.009524,-0.009524,10.4000,7875.0,11,3,0.003007,0.005550,-0.012531,-0.015074
9,10001,2002-04-01,0.008654,0.008654,10.4900,6850.0,11,3,-0.001037,-0.001189,0.009691,0.009843


In [64]:
# check the size of the merged return data

con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT permno) AS n_permno,
    COUNT(vwretd) AS n_with_market_return,
    MIN(ret_date) AS min_date,
    MAX(ret_date) AS max_date
FROM ret_mkt
""").df()

,n_rows,n_permno,n_with_market_return,min_date,max_date
0,19786721,6105,19786721,2002-03-18,2024-12-31


In [65]:
# assign trading day index within each PERMNO

con.execute("""
CREATE OR REPLACE TEMP TABLE ret_ranked AS
SELECT
    *,
    ROW_NUMBER() OVER (
        PARTITION BY permno
        ORDER BY ret_date
    ) AS trade_day_id
FROM ret_mkt
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [66]:
# check trading day index

con.execute("""
SELECT
    permno,
    ret_date,
    ret,
    vwretd,
    abret_vw,
    trade_day_id
FROM ret_ranked
ORDER BY permno, ret_date
LIMIT 20
""").df()

,permno,ret_date,ret,vwretd,abret_vw,trade_day_id
0,10001,2002-03-18,0.026569,0.001273,0.025296,1
1,10001,2002-03-19,-0.001623,0.003372,-0.004995,2
2,10001,2002-03-20,0.003405,-0.014411,0.017816,3
3,10001,2002-03-21,-0.026655,0.003144,-0.029799,4
4,10001,2002-03-22,0.027424,-0.004248,0.031672,5
5,10001,2002-03-25,-0.013346,-0.014346,0.001000,6
6,10001,2002-03-26,0.004783,0.005784,-0.001001,7
7,10001,2002-03-27,0.009664,0.005506,0.004158,8
8,10001,2002-03-28,-0.009524,0.003007,-0.012531,9
9,10001,2002-04-01,0.008654,-0.001037,0.009691,10


In [70]:
# find event day 0 for each candidate PERMNO
# event day 0 must be within 3 calendar days after the filing date

con.execute("""
CREATE OR REPLACE TEMP TABLE event0_candidates AS
SELECT
    sl.*,
    r0.ret_date AS event_date,
    r0.trade_day_id AS event_trade_day_id,
    r0.shrcd,
    r0.exchcd,
    r0.prc,
    r0.vol,
    r0.ret AS ret_event0
FROM sample_linked AS sl
LEFT JOIN LATERAL (
    SELECT
        rr.ret_date,
        rr.trade_day_id,
        rr.shrcd,
        rr.exchcd,
        rr.prc,
        rr.vol,
        rr.ret
    FROM ret_ranked AS rr
    WHERE rr.permno = sl.permno
      AND rr.ret_date >= sl.filing_dt
      AND rr.ret_date <= sl.filing_dt + INTERVAL 3 DAY
    ORDER BY rr.ret_date
    LIMIT 1
) AS r0 ON true
""")

In [71]:
# check event-date matching

con.execute("""
SELECT
    COUNT(DISTINCT row_id) AS n_filings,
    COUNT(DISTINCT CASE WHEN permno IS NOT NULL THEN row_id END) AS n_with_permno,
    COUNT(DISTINCT CASE WHEN event_date IS NOT NULL THEN row_id END) AS n_with_event_date,
    COUNT(DISTINCT CASE WHEN event_date IS NOT NULL THEN row_id END) * 1.0
        / COUNT(DISTINCT row_id) AS event_match_rate
FROM event0_candidates
""").df()

,n_filings,n_with_permno,n_with_event_date,event_match_rate
0,33236,13757,13659,0.41097


In [72]:
# check gap between filing date and event date

con.execute("""
SELECT
    event_date - filing_dt AS calendar_day_gap,
    COUNT(DISTINCT row_id) AS n_filings
FROM event0_candidates
WHERE event_date IS NOT NULL
GROUP BY event_date - filing_dt
ORDER BY calendar_day_gap
""").df()

,calendar_day_gap,n_filings
0,0,13560
1,1,1
2,3,98


In [73]:
# check how many filings have event dates found

con.execute("""
SELECT
    COUNT(DISTINCT row_id) AS n_filings,
    COUNT(DISTINCT CASE WHEN permno IS NOT NULL THEN row_id END) AS n_with_permno,
    COUNT(DISTINCT CASE WHEN event_date IS NOT NULL THEN row_id END) AS n_with_event_date,
    COUNT(DISTINCT CASE WHEN event_date IS NOT NULL THEN row_id END) * 1.0
        / COUNT(DISTINCT row_id) AS event_match_rate
FROM event0_candidates
""").df()

,n_filings,n_with_permno,n_with_event_date,event_match_rate
0,33236,13757,13659,0.41097


In [74]:
# choose one best PERMNO for each filing

con.execute("""
CREATE OR REPLACE TEMP TABLE event0 AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY row_id
            ORDER BY
                CASE WHEN event_date IS NOT NULL THEN 0 ELSE 1 END,
                CASE WHEN shrcd IN (10, 11) THEN 0 ELSE 1 END,
                CASE WHEN exchcd IN (1, 2, 3) THEN 0 ELSE 1 END,
                ABS(prc) DESC NULLS LAST,
                permno
        ) AS permno_rank
    FROM event0_candidates
) ranked
WHERE permno_rank = 1
""")

In [ ]:
# check the size of the final event0 table

con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT row_id) AS n_filings,
    COUNT(DISTINCT CASE WHEN permno IS NOT NULL THEN row_id END) AS n_with_permno,
    COUNT(DISTINCT CASE WHEN event_date IS NOT NULL THEN row_id END) AS n_with_event_date
FROM event0
""").df()

,n_rows,n_filings,n_with_permno,n_with_event_date
0,33236,33236,13757,13659


In [ ]:
# check gap between filing date and event date in the final event0 table

con.execute("""
SELECT
    event_date - filing_dt AS calendar_day_gap,
    COUNT(*) AS n_filings
FROM event0
WHERE event_date IS NOT NULL
GROUP BY event_date - filing_dt
ORDER BY calendar_day_gap
""").df()

,calendar_day_gap,n_filings
0,0,13560
1,1,1
2,3,98


In [77]:
# get event-window returns around event day 0

con.execute("""
CREATE OR REPLACE TEMP TABLE event_returns AS
SELECT
    e.*,
    rr.ret_date,
    rr.trade_day_id - e.event_trade_day_id AS event_day,
    rr.ret,
    rr.retx,
    rr.vwretd,
    rr.ewretd,
    rr.abret_vw,
    rr.abret_ew
FROM event0 AS e
LEFT JOIN ret_ranked AS rr
    ON e.permno = rr.permno
   AND rr.trade_day_id BETWEEN e.event_trade_day_id - 2
                           AND e.event_trade_day_id + 2
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# check the event returns data

con.execute("""
SELECT
    row_id,
    cik,
    name,
    filing_date,
    permno,
    event_date,
    ret_date,
    event_day,
    ret,
    vwretd,
    abret_vw
FROM event_returns
WHERE event_date IS NOT NULL
ORDER BY row_id, event_day
LIMIT 50
""").df()

,row_id,cik,name,filing_date,permno,event_date,ret_date,event_day,ret,vwretd,abret_vw
0,7,0001051343,COMMUNITY WEST BANCSHARES /,2009-03-30,84215,2009-03-30,2009-03-26,-2,-0.012766,0.025297,-0.038063
1,7,0001051343,COMMUNITY WEST BANCSHARES /,2009-03-30,84215,2009-03-30,2009-03-27,-1,0.032328,-0.021626,0.053954
2,7,0001051343,COMMUNITY WEST BANCSHARES /,2009-03-30,84215,2009-03-30,2009-03-30,0,-0.035491,-0.036280,0.000789
3,7,0001051343,COMMUNITY WEST BANCSHARES /,2009-03-30,84215,2009-03-30,2009-03-31,1,0.142857,0.014267,0.128590
4,7,0001051343,COMMUNITY WEST BANCSHARES /,2009-03-30,84215,2009-03-30,2009-04-01,2,-0.090909,0.017239,-0.108148
5,11,0001789029,"Aeva Technologies, Inc.",2024-03-15,19224,2024-03-15,2024-03-13,-2,-0.063636,-0.000746,-0.062890
6,11,0001789029,"Aeva Technologies, Inc.",2024-03-15,19224,2024-03-15,2024-03-14,-1,-0.053786,-0.004847,-0.048939
7,11,0001789029,"Aeva Technologies, Inc.",2024-03-15,19224,2024-03-15,2024-03-15,0,0.018674,-0.005012,0.023686
8,11,0001789029,"Aeva Technologies, Inc.",2024-03-15,19224,2024-03-15,2024-03-18,1,0.017325,0.004593,0.012732
9,11,0001789029,"Aeva Technologies, Inc.",2024-03-15,19224,2024-03-15,2024-03-19,2,0.029703,0.005022,0.024681


In [79]:
# check event-window completeness

con.execute("""
SELECT
    COUNT(DISTINCT row_id) AS n_filings,
    COUNT(DISTINCT CASE WHEN event_date IS NOT NULL THEN row_id END) AS n_with_event_date,
    COUNT(DISTINCT CASE WHEN event_day BETWEEN -2 AND 2 THEN row_id END) AS n_with_any_window_return
FROM event_returns
""").df()

,n_filings,n_with_event_date,n_with_any_window_return
0,33236,13659,13659


In [ ]:
# distribution of available return days in event windows

con.execute("""
SELECT
    n_ret_m2_p2,
    COUNT(*) AS n_filings
FROM (
    SELECT
        row_id,
        COUNT(CASE WHEN event_day BETWEEN -2 AND 2 THEN ret END) AS n_ret_m2_p2
    FROM event_returns
    GROUP BY row_id
)
GROUP BY n_ret_m2_p2
ORDER BY n_ret_m2_p2
""").df()

,n_ret_m2_p2,n_filings
0,0,19577
1,3,8
2,4,6
3,5,13645


In [ ]:
# aggregate event-window returns to filing-level CAR variables

out_path = ROOT / "data/generated/CAR/10k_sample_with_car.csv"

con.execute(f"""
COPY (
    SELECT
        row_id,
        cik,
        name,
        tickers,
        exchanges,
        entity_type,
        sic,
        state_of_incorporation,
        filing_date,
        report_date,
        url,

        permno,
        permco,
        gvkey,
        linktype,
        linkdt,
        linkenddt,
        event_date,

        -- event-day returns
        SUM(CASE WHEN event_day = 0 THEN ret ELSE NULL END) AS ret_0,
        SUM(CASE WHEN event_day = 0 THEN vwretd ELSE NULL END) AS mktret_vw_0,
        SUM(CASE WHEN event_day = 0 THEN ewretd ELSE NULL END) AS mktret_ew_0,
        SUM(CASE WHEN event_day = 0 THEN abret_vw ELSE NULL END) AS abret_vw_0,
        SUM(CASE WHEN event_day = 0 THEN abret_ew ELSE NULL END) AS abret_ew_0,

        -- raw cumulative returns
        SUM(CASE WHEN event_day BETWEEN 0 AND 1 THEN ret ELSE NULL END) AS rawret_0_p1,
        SUM(CASE WHEN event_day BETWEEN -1 AND 1 THEN ret ELSE NULL END) AS rawret_m1_p1,
        SUM(CASE WHEN event_day BETWEEN -2 AND 2 THEN ret ELSE NULL END) AS rawret_m2_p2,

        -- value-weighted market-adjusted CAR
        SUM(CASE WHEN event_day BETWEEN 0 AND 1 THEN abret_vw ELSE NULL END) AS car_vw_0_p1,
        SUM(CASE WHEN event_day BETWEEN -1 AND 1 THEN abret_vw ELSE NULL END) AS car_vw_m1_p1,
        SUM(CASE WHEN event_day BETWEEN -2 AND 2 THEN abret_vw ELSE NULL END) AS car_vw_m2_p2,

        -- equal-weighted market-adjusted CAR, optional
        SUM(CASE WHEN event_day BETWEEN 0 AND 1 THEN abret_ew ELSE NULL END) AS car_ew_0_p1,
        SUM(CASE WHEN event_day BETWEEN -1 AND 1 THEN abret_ew ELSE NULL END) AS car_ew_m1_p1,
        SUM(CASE WHEN event_day BETWEEN -2 AND 2 THEN abret_ew ELSE NULL END) AS car_ew_m2_p2,

        -- completeness checks
        COUNT(CASE WHEN event_day = 0 THEN ret END) AS n_ret_0,
        COUNT(CASE WHEN event_day BETWEEN 0 AND 1 THEN ret END) AS n_ret_0_p1,
        COUNT(CASE WHEN event_day BETWEEN -1 AND 1 THEN ret END) AS n_ret_m1_p1,
        COUNT(CASE WHEN event_day BETWEEN -2 AND 2 THEN ret END) AS n_ret_m2_p2

    FROM event_returns
    GROUP BY
        row_id,
        cik,
        name,
        tickers,
        exchanges,
        entity_type,
        sic,
        state_of_incorporation,
        filing_date,
        report_date,
        url,
        permno,
        permco,
        gvkey,
        linktype,
        linkdt,
        linkenddt,
        event_date
) TO '{out_path}' WITH (HEADER, DELIMITER ',');
""")

print("Saved to:", out_path)

Saved to: ../../data/generated/10k_sample_with_car.csv
